In [ ]:
# ==== Imports ====
import numpy as np
import glob
import os
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from tifffile import imread, imwrite
import random


In [ ]:
# ==== Parameters ====

# --- Input/Output Folders ---
RAW_FOLDER = "raws"
FLATFIELD_FOLDER = "flats"
OUTPUT_BASE = "output"

# --- Processing Settings ---
NUM_IMAGES = 50                      # Number of random raw images to process
MIN_CONTOUR_AREA = 800              # Minimum contour area to consider for analysis

# --- Rotation Settings ---
ROTATION_CENTER = None       # Rotation center (x, y) coordinates
ROTATION_ANGLE = 0               # Rotation angle in degrees (positive = acw)


In [ ]:
# ==== Flatfield Correction ====

def flatfield_correction(raw_folder, flatfield_folder, output_folder, num_images=None, flip_horizontal=True):
    os.makedirs(output_folder, exist_ok=True)
    flatfield_files = sorted(glob.glob(os.path.join(flatfield_folder, '*.tif')))
    flatfields = [imread(f).astype(np.float32) for f in flatfield_files]
    flatfield_avg = np.mean(flatfields, axis=0)

    raw_files = sorted(glob.glob(os.path.join(raw_folder, '*.tif')))
    if num_images is not None and num_images < len(raw_files):
        raw_files = random.sample(raw_files, num_images)

    for idx, raw_path in enumerate(raw_files, 1):
        raw_img = imread(raw_path).astype(np.float32)
        if raw_img.shape != flatfield_avg.shape:
            print(f"Shape mismatch: {raw_path} - Skipping")
            continue

        corrected = raw_img - flatfield_avg
        low, high = np.percentile(corrected, (1, 99))
        corrected_clipped = np.clip(corrected, low, high)
        corrected_norm = (corrected_clipped - low) / (high - low) * 255
        corrected_inv = 255 - corrected_norm.astype(np.uint8)

        if flip_horizontal:
            corrected_inv = cv2.flip(corrected_inv, 1)

        out_path = os.path.join(output_folder, os.path.basename(raw_path))
        imwrite(out_path, corrected_inv)
        print(f"\rProcessing {idx}/{len(raw_files)} images", end='', flush=True)

    print("\nFlatfield correction complete.")


In [ ]:
# ==== Grayscale Conversion ====

def grayscale_conversion(input_folder, output_folder, rotation_center=None, rotation_angle=0):
    os.makedirs(output_folder, exist_ok=True)
    image_files = glob.glob(os.path.join(input_folder, '*.tif'))

    for idx, img_path in enumerate(image_files, 1):
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            print(f"Error reading: {img_path}")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

        # --- Apply rotation if needed ---
        if rotation_center is not None and rotation_angle != 0:
            M = cv2.getRotationMatrix2D(rotation_center, rotation_angle, 1.0)
            gray = cv2.warpAffine(gray, M, (gray.shape[1], gray.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

        # --- Normalize and Save ---
        stretched = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
        out_path = os.path.join(output_folder, os.path.basename(img_path))
        cv2.imwrite(out_path, stretched)

        print(f"\rGrayscale {idx}/{len(image_files)}", end='', flush=True)

    print("\nGrayscale conversion complete.")


In [ ]:
# ==== Contour Detection for L_b ====

def analyze_spray_breakup(
    input_folder, 
    output_folder, 
    output_csv, 
    min_contour_area=500,
    plot_folder=None
):
    os.makedirs(output_folder, exist_ok=True)
    image_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.tif')]
    results = []

    for fname in image_files:
        img_path = os.path.join(input_folder, fname)
        gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if gray is None:
            print(f"Error reading: {img_path}")
            continue

        # --- Binary mask & clean-up ---
        _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        cleaned_binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # --- Find contours ---
        contours, _ = cv2.findContours(cleaned_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"No contours found in {fname}")
            results.append({
                'filename': fname,
                'contour_area': 0,
                'bounding_box_x': np.nan,
                'bounding_box_y': np.nan,
                'bounding_box_width': np.nan,
                'bounding_box_height': np.nan
            })
            continue

        # --- Largest contour ---
        main_contour = max(contours, key=cv2.contourArea)
        contour_area = cv2.contourArea(main_contour)

        if contour_area < min_contour_area:
            print(f"Main contour too small in {fname}")
            results.append({
                'filename': fname,
                'contour_area': contour_area,
                'bounding_box_x': np.nan,
                'bounding_box_y': np.nan,
                'bounding_box_width': np.nan,
                'bounding_box_height': np.nan
            })
            continue

        x, y, w, h = cv2.boundingRect(main_contour)

        # --- Save result ---
        results.append({
            'filename': fname,
            'contour_area': contour_area,
            'bounding_box_x': x,
            'bounding_box_y': y,
            'bounding_box_width': w,
            'bounding_box_height': h
        })

        # --- Visualization ---
        vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 255, 0), 1)
        cv2.drawContours(vis, [main_contour], -1, (255, 0, 0), 1)

        out_path = os.path.join(output_folder, fname)
        cv2.imwrite(out_path, vis)

    # --- Save to CSV ---
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Contour analysis complete. Results saved to {output_csv}")


In [ ]:
# ==== Full Processing Pipeline ====

def process_spray_images():
    corrected = os.path.join(OUTPUT_BASE, "flatfield_corrected")
    grayscale = os.path.join(OUTPUT_BASE, "grayscale")
    analysis = os.path.join(OUTPUT_BASE, "spray_analysis")
    plots = os.path.join(OUTPUT_BASE, "intensity_profiles")

    print(f"Processing {NUM_IMAGES} random images...")

    flatfield_correction(RAW_FOLDER, FLATFIELD_FOLDER, corrected, NUM_IMAGES, flip_horizontal=False)
    grayscale_conversion(corrected, grayscale, rotation_center=ROTATION_CENTER, rotation_angle=ROTATION_ANGLE)
    analyze_spray_breakup(
        input_folder=grayscale,
        output_folder=analysis,
        output_csv=os.path.join(OUTPUT_BASE, "breakup_results.csv"),
        min_contour_area=MIN_CONTOUR_AREA,
    )
    print("Processing complete!")


In [ ]:
# ==== Run Everything ====
process_spray_images()
